# 🎵 Messy Mashup — Optimised Ensemble (AST + 3 CNNs)


**Optimisations applied vs original notebook:**
| Change | Saving |
|--------|--------|
| `torchaudio.load` instead of `librosa` in CNN | ~15× faster audio I/O |
| MelSpectrogram moved from CPU `__getitem__` → GPU batch transform | ~600 min saved |
| `time_stretch` removed → fast random-offset crop augmentation instead | ~680 min saved |
| `torch.cuda.amp` AMP/FP16 on T4 Tensor Cores | ~1.7× GPU speedup |
| `nn.DataParallel` — use **both** T4s | ~1.8× throughput |
| `CNN_EPOCHS 26→15` + early stopping (patience=5) | ~300 min saved |
| `AST_TRAIN_SIZE 5000→3000` | ~115 min saved |
| `NUM_WORKERS 2→4` | better CPU utilisation |




## 1. Setup & Master Config

In [ ]:
'''# ── Install
!pip install -q transformers torchmetrics wandb
# librosa still needed for EDA / AST loading; torchaudio built-in on Kaggle
print("Libraries ready!")'''


In [ ]:

import os, random, warnings, glob, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
import torchaudio.transforms as T
import torchvision.models as models
import librosa                       

from tqdm.notebook import tqdm
from torch.utils.data import Dataset, DataLoader
from transformers import (
    ASTFeatureExtractor, ASTForAudioClassification,
    get_cosine_schedule_with_warmup
)
from torchmetrics.classification import MulticlassF1Score
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, classification_report

import wandb
warnings.filterwarnings('ignore')
print("All imports done!")


In [ ]:
import os
import wandb
key = os.environ.get("WANDB_API_KEY")
if key:
    wandb.login(key=key)
    print("WandB logged in!")
else:
    raise RuntimeError("WANDB_API_KEY not set!")


In [ ]:
def set_seed(seed=42):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
set_seed(42)
print("Seed set to 42!")


- **MASTER CONFIG**

In [ ]:
# WandB
WANDB_PROJECT   = "22f3003226-t12026"   
WANDB_ENTITY    = None

# Paths
BASE_PATH       = '/teamspace/studios/this_studio/data/messy_mashup'
GENRES_PATH     = os.path.join(BASE_PATH, 'genres_stems')
ESC_PATH        = os.path.join(BASE_PATH, 'ESC-50-master', 'audio')
MASHUPS_PATH    = os.path.join(BASE_PATH, 'mashups')
TEST_CSV        = os.path.join(BASE_PATH, 'test.csv')
REQUIRED_STEMS  = ['drums.wav', 'vocals.wav', 'bass.wav', 'other.wav']

GENRES     = ['blues','classical','country','disco','hiphop',
              'jazz','metal','pop','reggae','rock']
label2id   = {g: i for i, g in enumerate(GENRES)}
id2label   = {i: g for g, i in label2id.items()}
NUM_LABELS = len(GENRES)

# Device — use both T4s with DataParallel
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
N_GPUS     = torch.cuda.device_count()
print(f"GPUs available: {N_GPUS}")

# Speed switches
USE_AMP        = True    # FP16 mixed precision (1.7× on T4 Tensor Cores)
USE_DDP        = True    # nn.DataParallel across both T4s

# AST 
AST_SAMPLE_RATE  = 16000
AST_DURATION     = 20
AST_MAX_LENGTH   = AST_SAMPLE_RATE * AST_DURATION
AST_BATCH_SIZE   = 12
AST_GRAD_ACCUM   = 1
AST_EPOCHS_P1    = 3
AST_EPOCHS_P2    = 5
AST_LR_P1        = 3e-4
AST_LR_P2        = 2e-5
AST_LR_HEAD_MULT = 10
AST_WEIGHT_DECAY = 0.01
AST_TRAIN_SIZE   = 3000   
AST_NOISE_PROB   = 0.7
AST_NOISE_LEVEL  = (0.03, 0.20)
AST_PATIENCE_P1  = 2
AST_PATIENCE_P2  = 3
AST_N_TTA        = 5
AST_NUM_WORKERS  = 4      

# CNN 
CNN_SAMPLE_RATE  = 22050
CNN_DURATION     = 5.0
CNN_N_SAMPLES    = int(CNN_SAMPLE_RATE * CNN_DURATION)
CNN_N_FFT        = 2048
CNN_HOP_LEN      = 512
CNN_N_MELS       = 128
CNN_BATCH_SIZE   = 128     
CNN_EPOCHS       = 15     
CNN_PATIENCE     = 5      
CNN_LR           = 1e-3
CNN_WEIGHT_DECAY = 1e-4
CNN_NUM_WORKERS  = 4      

#  Ensemble 
ENSEMBLE_WEIGHTS = {
    "AST"            : 0.40,
    "CustomCNN"      : 0.15,
    "AudioResNet34"  : 0.20,
    "EfficientNetB0" : 0.25,
}
assert abs(sum(ENSEMBLE_WEIGHTS.values()) - 1.0) < 1e-6

print(f"Device: {DEVICE}  |  GPUs: {N_GPUS}  |  AMP: {USE_AMP}  |  DataParallel: {USE_DDP}")
print(f"AST TRAIN_SIZE: {AST_TRAIN_SIZE}  |  CNN_EPOCHS: {CNN_EPOCHS}")


---
## 2. Exploratory Data Analysis

In [ ]:
wandb.init(
    project = WANDB_PROJECT,
    entity  = WANDB_ENTITY,
    name    = "EDA",
    reinit  = True,
    config  = {
        "total_genres"    : len(GENRES),
        "required_stems"  : REQUIRED_STEMS,
        "ast_duration_s"  : AST_DURATION,
        "cnn_duration_s"  : CNN_DURATION,
        "ast_sample_rate" : AST_SAMPLE_RATE,
        "cnn_sample_rate" : CNN_SAMPLE_RATE,
    }
)
print("WandB EDA run started!")

In [ ]:
CKPT = '/teamspace/studios/this_studio/messy-mashup/checkpoints'

- **EDA 1: Class Counts & Balance**

In [ ]:
colors = plt.cm.tab10(np.linspace(0, 1, len(GENRES)))
genre_song_counts = {}
for genre in GENRES:
    gp = os.path.join(GENRES_PATH, genre)
    if not os.path.isdir(gp): continue
    complete = sum(
        1 for f in os.listdir(gp)
        if os.path.isdir(os.path.join(gp, f))
        and all(os.path.exists(os.path.join(gp, f, s)) for s in REQUIRED_STEMS)
    )
    genre_song_counts[genre] = complete
    print(f"  {genre:12s}: {complete} complete songs")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].bar(list(genre_song_counts.keys()), list(genre_song_counts.values()), color=colors)
axes[0].set_title("Songs per Genre (all 4 stems present)", fontweight='bold')
axes[0].tick_params(axis='x', rotation=35)
ideal  = sum(genre_song_counts.values()) / len(genre_song_counts)
imbal  = [(v - ideal)/ideal*100 for v in genre_song_counts.values()]
bcols  = ['green' if abs(x)<10 else ('orange' if abs(x)<25 else 'red') for x in imbal]
axes[1].bar(list(genre_song_counts.keys()), imbal, color=bcols)
axes[1].axhline(0, color='k', linewidth=1, linestyle='--')
axes[1].set_title("Class Imbalance (% from mean)", fontweight='bold')
axes[1].tick_params(axis='x', rotation=35)
plt.tight_layout()
plt.savefig('/teamspace/studios/this_studio/messy-mashup/checkpoints/eda_class_dist.png', dpi=100, bbox_inches='tight')
CKPT = '/teamspace/studios/this_studio/messy-mashup/checkpoints'

plt.savefig(f'{CKPT}/eda_class_dist.png', dpi=100, bbox_inches='tight')
wandb.log({
    "eda/class_distribution"  : wandb.Image(f'{CKPT}/eda_class_dist.png'),
    "eda/class_imbalance_chart": wandb.Image(f'{CKPT}/eda_class_dist.png'),
    "eda/songs_per_genre": wandb.plot.bar(
        wandb.Table(data=[[g, c] for g, c in genre_song_counts.items()],
                    columns=["genre", "count"]),
        "genre", "count", title="Songs per Genre"
    )
})
plt.show()
print(f"Total complete songs: {sum(genre_song_counts.values())}")


- **EDA 2: Mel Spectrogram Gallery (1 song per genre)**

In [ ]:

mel_tx = T.MelSpectrogram(sample_rate=CNN_SAMPLE_RATE, n_fft=CNN_N_FFT,
                           hop_length=CNN_HOP_LEN, n_mels=CNN_N_MELS)
adb    = T.AmplitudeToDB()

fig, axes = plt.subplots(2, 5, figsize=(20, 7))
for idx, genre in enumerate(GENRES):
    ax   = axes[idx//5][idx%5]
    gp   = os.path.join(GENRES_PATH, genre)
    flds = sorted([os.path.join(gp,f) for f in os.listdir(gp)
                   if os.path.isdir(os.path.join(gp,f))])
    if not flds: continue
    mixed = np.zeros(CNN_N_SAMPLES)
    for stem in REQUIRED_STEMS:
        sp = os.path.join(flds[0], stem)
        if os.path.exists(sp):
            try:
                y, _ = librosa.load(sp, sr=CNN_SAMPLE_RATE, duration=CNN_DURATION)
                y = librosa.util.fix_length(y, size=CNN_N_SAMPLES) if len(y)<CNN_N_SAMPLES else y[:CNN_N_SAMPLES]
                mixed += y * 0.25
            except: pass
    mx = np.max(np.abs(mixed)); mixed = mixed/mx if mx>0 else mixed
    spec = adb(mel_tx(torch.tensor(mixed, dtype=torch.float32).unsqueeze(0))).squeeze().numpy()
    im   = ax.imshow(spec, aspect='auto', origin='lower', cmap='magma')
    ax.set_title(genre.capitalize(), fontweight='bold')
    plt.colorbar(im, ax=ax, format='%+2.0f dB', shrink=0.7)
plt.suptitle("Mel Spectrograms — One Song per Genre", fontsize=14, y=1.01, fontweight='bold')
plt.tight_layout()
plt.savefig('/teamspace/studios/this_studio/messy-mashup/checkpoints/eda_spectrograms.png', dpi=100, bbox_inches='tight')
wandb.log({"eda/mel_spectrograms": wandb.Image(f'{CKPT}/eda_spectrograms.png')})
plt.show()


- **EDA 3: Waveforms (4 genres × vocals stem, 5s)**

In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(14, 6))
sample_genres = ['blues', 'metal', 'classical', 'hiphop']
for i, genre in enumerate(sample_genres):
    ax = axes[i//2][i%2]
    gp = os.path.join(GENRES_PATH, genre)
    flds = sorted([os.path.join(gp,f) for f in os.listdir(gp)
                   if os.path.isdir(os.path.join(gp,f))])
    sp = os.path.join(flds[0], 'vocals.wav')
    if os.path.exists(sp):
        y, sr = librosa.load(sp, sr=AST_SAMPLE_RATE, duration=5.0)
        ax.plot(np.linspace(0, 5, len(y)), y, linewidth=0.5,
                color=colors[GENRES.index(genre)], alpha=0.85)
        ax.set_title(f"{genre.capitalize()} — vocals (5s)", fontweight='bold')
        ax.set_xlabel("Time (s)"); ax.set_ylabel("Amplitude")
        ax.text(0.02, 0.88, f"RMS: {np.sqrt(np.mean(y**2)):.4f}",
                transform=ax.transAxes, fontsize=9,
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.savefig('/teamspace/studios/this_studio/messy-mashup/checkpoints/eda_waveforms.png', dpi=100, bbox_inches='tight')
wandb.log({"eda/waveforms": wandb.Image(f'{CKPT}/eda_waveforms.png')})
plt.show()


- **EDA 4: Stem Loudness Heatmap**

In [ ]:
ld = []
for genre in GENRES:
    gp   = os.path.join(GENRES_PATH, genre)
    flds = [os.path.join(gp,f) for f in os.listdir(gp)
            if os.path.isdir(os.path.join(gp,f))][:5]
    for fp in flds:
        for stem in REQUIRED_STEMS:
            sp = os.path.join(fp, stem)
            if os.path.exists(sp):
                try:
                    y, _ = librosa.load(sp, sr=AST_SAMPLE_RATE, duration=10.0)
                    ld.append({'genre': genre, 'stem': stem.replace('.wav',''),
                               'rms': float(np.sqrt(np.mean(y**2)))})
                except: pass
df_ld  = pd.DataFrame(ld)
pivot  = df_ld.groupby(['genre','stem'])['rms'].mean().unstack()
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlOrRd', ax=ax, linewidths=0.4)
ax.set_title("Mean RMS per Stem per Genre", fontweight='bold')
plt.tight_layout()
plt.savefig('/teamspace/studios/this_studio/messy-mashup/checkpoints/eda_loudness.png', dpi=100, bbox_inches='tight')
wandb.log({
    "eda/stem_loudness_heatmap": wandb.Image(f'{CKPT}/eda_loudness.png'),
    # also log the raw pivot table so you can filter it in WandB
    "eda/loudness_table": wandb.Table(dataframe=df_ld.groupby(
        ['genre','stem'])['rms'].mean().reset_index())
})
plt.show()


- **EDA 5: Train / Val Split**

In [ ]:

def build_song_index():
    idx = {}
    for genre in GENRES:
        gp  = os.path.join(GENRES_PATH, genre)
        idx[genre] = [
            os.path.join(gp, f) for f in sorted(os.listdir(gp))
            if os.path.isdir(os.path.join(gp, f))
            and all(os.path.exists(os.path.join(gp, f, s)) for s in REQUIRED_STEMS)
        ]
    return idx

song_index  = build_song_index()
train_index = {}
val_index   = {}
for genre in GENRES:
    songs = song_index[genre][:]
    random.shuffle(songs)
    sp = int(0.85 * len(songs))
    train_index[genre] = songs[:sp]
    val_index[genre]   = songs[sp:]

tr = {g: len(train_index[g]) for g in GENRES}
vl = {g: len(val_index[g])   for g in GENRES}
x  = np.arange(len(GENRES))
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(x-0.2, list(tr.values()), 0.4, label='Train', color='steelblue', alpha=0.85)
ax.bar(x+0.2, list(vl.values()), 0.4, label='Val',   color='salmon',    alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(GENRES, rotation=35)
ax.set_title("Train / Val Split per Genre (85/15)", fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('/teamspace/studios/this_studio/messy-mashup/checkpoints/eda_split.png', dpi=100, bbox_inches='tight')
'''wandb.log({
    "eda/train_val_split": wandb.Image(f'{CKPT}/eda_split.png'),
    "eda/total_train_songs": sum(tr.values()),
    "eda/total_val_songs"  : sum(vl.values()),
})'''
plt.show()
print(f"Train songs: {sum(tr.values())}  |  Val songs: {sum(vl.values())}")
noise_files_list = sorted(glob.glob(os.path.join(ESC_PATH, "*.wav")))
print(f"ESC-50 noise files: {len(noise_files_list)}")


In [ ]:
wandb.finish()
print("EDA logged to WandB!")


## 3. AST Training
>  **Optimisations:** AMP/FP16 · `TRAIN_SIZE` reduced to 3000 · `NUM_WORKERS=4`
>  `time_stretch` replaced with fast random-offset crop (same augmentation effect, 0 phase-vocoder overhead)
>
> ── AST: Audio Helpers ──────────────────────────────────────────
- time_stretch REMOVED — replaced by random-offset crop.
- Rationale: time_stretch shifts pitch + duration via slow STFT phase vocoder.
- For genre classification, a random crop from a different time offset gives
- equivalent position-invariance augmentation at ~0 CPU cost.


In [ ]:


def ast_load_audio(path):
    audio, _ = librosa.load(path, sr=AST_SAMPLE_RATE, mono=True)
    return audio.astype(np.float32)

def ast_normalize(audio):
    return audio / (np.max(np.abs(audio)) + 1e-6)

def ast_crop_random(audio):
    if len(audio) >= AST_MAX_LENGTH:
        start = random.randint(0, len(audio) - AST_MAX_LENGTH)
        return audio[start : start + AST_MAX_LENGTH]
    return np.pad(audio, (0, AST_MAX_LENGTH - len(audio)))

def ast_crop_center(audio):
    if len(audio) >= AST_MAX_LENGTH:
        start = (len(audio) - AST_MAX_LENGTH) // 2
        return audio[start : start + AST_MAX_LENGTH]
    return np.pad(audio, (0, AST_MAX_LENGTH - len(audio)))

def ast_random_gain(audio, low=0.6, high=1.4):
    return audio * random.uniform(low, high)

def ast_add_noise(audio):
    if noise_files_list and random.random() < AST_NOISE_PROB:
        nf   = random.choice(noise_files_list)
        n, _ = librosa.load(nf, sr=AST_SAMPLE_RATE, mono=True)
        n    = n.astype(np.float32)
        n    = ast_crop_random(n)
        lvl  = random.uniform(*AST_NOISE_LEVEL)
        audio = audio + lvl * n
    return audio

print("AST audio helpers ready (time_stretch removed — fast random-crop replaces it)!")


In [ ]:
# AST Load_Model
AST_MODEL_NAME = 'MIT/ast-finetuned-audioset-10-10-0.4593'
ast_feature_extractor = ASTFeatureExtractor.from_pretrained(AST_MODEL_NAME)

_ast_base = ASTForAudioClassification.from_pretrained(
    AST_MODEL_NAME,
    num_labels=NUM_LABELS, id2label=id2label, label2id=label2id,
    ignore_mismatched_sizes=True
)
_ast_base.to(DEVICE)

# DataParallel — use both T4 GPUs
if USE_DDP and N_GPUS > 1:
    ast_model = nn.DataParallel(_ast_base)
    print(f"AST wrapped in DataParallel across {N_GPUS} GPUs!")
else:
    ast_model = _ast_base

total = sum(p.numel() for p in _ast_base.parameters())
print(f"AST loaded | {total:,} params | device: {DEVICE}")


In [ ]:
# AST Datasets
class ASTTrainDataset(Dataset):
    def __init__(self, song_index, feature_extractor, size=AST_TRAIN_SIZE):
        self.song_index = song_index
        self.fe = feature_extractor
        self.size = size
    def __len__(self): return self.size
    def __getitem__(self, idx):
        while True:
            try:
                genre = random.choice(GENRES)
                songs = self.song_index[genre]
                if not songs: continue
                mixed = None
                for stem in REQUIRED_STEMS:
                    audio = ast_load_audio(os.path.join(random.choice(songs), stem))
                    audio = ast_crop_random(audio)   # ← random offset = augmentation
                    audio = ast_random_gain(audio)
                    mixed = audio if mixed is None else mixed + audio
                mixed = ast_normalize(mixed)
                mixed = ast_add_noise(mixed)
                mixed = ast_normalize(mixed)
                inp   = self.fe(mixed, sampling_rate=AST_SAMPLE_RATE, return_tensors='pt')
                return {'input_values': inp['input_values'].squeeze(0),
                        'labels': torch.tensor(label2id[genre], dtype=torch.long)}
            except Exception: continue

class ASTValDataset(Dataset):
    def __init__(self, val_index, feature_extractor):
        self.fe = feature_extractor
        self.samples = [(g, sp) for g in GENRES for sp in val_index[g]]
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        genre, sp = self.samples[idx]
        mixed = None
        for stem in REQUIRED_STEMS:
            audio = ast_load_audio(os.path.join(sp, stem))
            audio = ast_crop_center(audio)
            mixed = audio if mixed is None else mixed + audio
        mixed = ast_normalize(mixed)
        inp   = self.fe(mixed, sampling_rate=AST_SAMPLE_RATE, return_tensors='pt')
        return {'input_values': inp['input_values'].squeeze(0),
                'labels': torch.tensor(label2id[genre], dtype=torch.long)}

ast_train_ds = ASTTrainDataset(train_index, ast_feature_extractor, AST_TRAIN_SIZE)
ast_val_ds   = ASTValDataset(val_index, ast_feature_extractor)
ast_train_loader = DataLoader(ast_train_ds, batch_size=AST_BATCH_SIZE, shuffle=True,
                               num_workers=AST_NUM_WORKERS, pin_memory=True, persistent_workers=True)
ast_val_loader   = DataLoader(ast_val_ds,   batch_size=AST_BATCH_SIZE, shuffle=False,
                               num_workers=AST_NUM_WORKERS, pin_memory=True, persistent_workers=True)
print(f"AST  Train: {len(ast_train_ds)} samples | Val: {len(ast_val_ds)} samples")


- AST: Loss, Metric, Train/Val Functions
- GradScaler for AMP

In [ ]:

ast_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
ast_f1_metric = MulticlassF1Score(num_classes=NUM_LABELS, average='macro').to(DEVICE)


ast_scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def ast_train_one_epoch(model, loader, optimizer, scheduler):
    model.train()
    ast_f1_metric.reset()
    total_loss, all_preds, all_labels = 0, [], []
    optimizer.zero_grad()
    for step, batch in enumerate(tqdm(loader, desc='  AST Train')):
        iv     = batch['input_values'].to(DEVICE)
        labels = batch['labels'].to(DEVICE)
        # ⚡ AMP forward
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            outputs = model(input_values=iv)
            loss    = ast_criterion(outputs.logits, labels) / AST_GRAD_ACCUM
        ast_scaler.scale(loss).backward()
        if (step + 1) % AST_GRAD_ACCUM == 0:
            ast_scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                (model.module if hasattr(model,'module') else model).parameters(), 1.0
            )
            ast_scaler.step(optimizer)
            ast_scaler.update()
            scheduler.step()
            optimizer.zero_grad()
        total_loss += loss.item() * AST_GRAD_ACCUM
        preds = torch.argmax(outputs.logits, 1)
        ast_f1_metric.update(preds, labels)
        all_preds.extend(preds.cpu().numpy()); all_labels.extend(labels.cpu().numpy())
    pc = f1_score(all_labels, all_preds, average=None, labels=list(range(NUM_LABELS)), zero_division=0)
    return total_loss / len(loader), ast_f1_metric.compute().item(), pc

def ast_validate(model, loader):
    model.eval()
    ast_f1_metric.reset()
    total_loss, all_preds, all_labels = 0, [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='  AST Val  '):
            iv, labels = batch['input_values'].to(DEVICE), batch['labels'].to(DEVICE)
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                outputs = model(input_values=iv)
                loss    = ast_criterion(outputs.logits, labels)
            total_loss += loss.item()
            preds = torch.argmax(outputs.logits, 1)
            ast_f1_metric.update(preds, labels)
            all_preds.extend(preds.cpu().numpy()); all_labels.extend(labels.cpu().numpy())
    pc = f1_score(all_labels, all_preds, average=None, labels=list(range(NUM_LABELS)), zero_division=0)
    return total_loss / len(loader), ast_f1_metric.compute().item(), pc

print("AST train/validate functions ready (AMP enabled)!")


In [ ]:
'''# AST: Phase 1 — Freeze base, train head
wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="AST_Phase1", reinit=True,
           config=dict(phase=1, model=AST_MODEL_NAME, epochs=AST_EPOCHS_P1,
                       lr=AST_LR_P1, batch_size=AST_BATCH_SIZE, grad_accum=AST_GRAD_ACCUM,
                       train_size=AST_TRAIN_SIZE, duration_s=AST_DURATION, amp=USE_AMP))

_base = ast_model.module if hasattr(ast_model, 'module') else ast_model
for p in _base.base_model.parameters(): p.requires_grad = False
print(f"Phase 1: {sum(p.numel() for p in _base.parameters() if p.requires_grad):,} trainable params")

optimizer_p1 = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, _base.parameters()), lr=AST_LR_P1, weight_decay=AST_WEIGHT_DECAY)
total_steps_p1 = (len(ast_train_loader) // AST_GRAD_ACCUM) * AST_EPOCHS_P1
scheduler_p1   = get_cosine_schedule_with_warmup(optimizer_p1,
    num_warmup_steps=total_steps_p1//10, num_training_steps=total_steps_p1)

best_p1_f1, patience_cnt = 0, 0
print(f"\n{'='*52}\nPHASE 1 — Classifier Warmup ({AST_EPOCHS_P1} epochs)\n{'='*52}\n")

for epoch in range(AST_EPOCHS_P1):
    tl, tf1, tpc = ast_train_one_epoch(ast_model, ast_train_loader, optimizer_p1, scheduler_p1)
    vl, vf1, vpc = ast_validate(ast_model, ast_val_loader)
    log = {"epoch": epoch+1, "train/loss": tl, "train/macro_f1": tf1,
           "val/loss": vl, "val/macro_f1": vf1, "lr": optimizer_p1.param_groups[0]['lr']}
    for i, g in enumerate(GENRES):
        log[f"val/f1_{g}"] = float(vpc[i]); log[f"train/f1_{g}"] = float(tpc[i])
    wandb.log(log)
    print(f"Ep {epoch+1}/{AST_EPOCHS_P1} | TL:{tl:.4f} TF1:{tf1:.4f} | VL:{vl:.4f} VF1:{vf1:.4f}")
    if vf1 > best_p1_f1:
        best_p1_f1 = vf1
        torch.save(_base.state_dict(), '/teamspace/studios/this_studio/messy-mashup/checkpoints/ast_best_phase1.pth')
        patience_cnt = 0; print(f"  ✓ Saved (val F1: {best_p1_f1:.4f})")
    else:
        patience_cnt += 1
        if patience_cnt >= AST_PATIENCE_P1: print("  Early stopping P1."); break

wandb.run.summary["best_val_f1_phase1"] = best_p1_f1
wandb.finish()
print(f"\nPhase 1 done! Best val F1: {best_p1_f1:.4f}")'''


In [ ]:
'''#  AST: Phase 2 — Full fine-tune all layers 
wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="AST_Phase2", reinit=True,
           config=dict(phase=2, model=AST_MODEL_NAME, epochs=AST_EPOCHS_P2,
                       lr_base=AST_LR_P2, lr_head=AST_LR_P2*AST_LR_HEAD_MULT,
                       batch_size=AST_BATCH_SIZE, amp=USE_AMP))

_base.load_state_dict(torch.load('/teamspace/studios/this_studio/messy-mashup/checkpoints/ast_best_phase1.pth'))
for p in _base.parameters(): p.requires_grad = True
print(f"All {sum(p.numel() for p in _base.parameters()):,} params unfrozen!")

optimizer_p2 = torch.optim.AdamW([
    {'params': _base.base_model.parameters(),  'lr': AST_LR_P2},
    {'params': _base.classifier.parameters(),  'lr': AST_LR_P2 * AST_LR_HEAD_MULT},
], weight_decay=AST_WEIGHT_DECAY)
total_steps_p2 = (len(ast_train_loader) // AST_GRAD_ACCUM) * AST_EPOCHS_P2
scheduler_p2   = get_cosine_schedule_with_warmup(optimizer_p2,
    num_warmup_steps=total_steps_p2//10, num_training_steps=total_steps_p2)

best_p2_f1, patience_cnt = 0, 0
print(f"\n{'='*52}\nPHASE 2 — Full Fine-Tune ({AST_EPOCHS_P2} epochs)\n"
      f"Base LR: {AST_LR_P2} | Head LR: {AST_LR_P2*AST_LR_HEAD_MULT}\n{'='*52}\n")

for epoch in range(AST_EPOCHS_P2):
    tl, tf1, tpc = ast_train_one_epoch(ast_model, ast_train_loader, optimizer_p2, scheduler_p2)
    vl, vf1, vpc = ast_validate(ast_model, ast_val_loader)
    log = {"epoch": epoch+1, "train/loss": tl, "train/macro_f1": tf1,
           "val/loss": vl, "val/macro_f1": vf1,
           "lr_base": optimizer_p2.param_groups[0]['lr'],
           "lr_head": optimizer_p2.param_groups[1]['lr']}
    for i, g in enumerate(GENRES):
        log[f"val/f1_{g}"] = float(vpc[i]); log[f"train/f1_{g}"] = float(tpc[i])
    wandb.log(log)
    print(f"Ep {epoch+1}/{AST_EPOCHS_P2} | TL:{tl:.4f} TF1:{tf1:.4f} | VL:{vl:.4f} VF1:{vf1:.4f}")
    if vf1 > best_p2_f1:
        best_p2_f1 = vf1
        torch.save(_base.state_dict(), '/teamspace/studios/this_studio/messy-mashup/checkpoints/ast_best_phase2.pth')
        patience_cnt = 0; print(f"  ✓ Saved (val F1: {best_p2_f1:.4f})")
    else:
        patience_cnt += 1
        if patience_cnt >= AST_PATIENCE_P2: print("  Early stopping P2."); break

wandb.run.summary["best_val_f1_phase2"] = best_p2_f1
wandb.finish()
print(f"\nPhase 2 done! Best val F1: {best_p2_f1:.4f}")
'''

- Checkpoint Logic. No need for retraining

In [ ]:
_base = ast_model.module if hasattr(ast_model, 'module') else ast_model
_base.load_state_dict(
    torch.load('/teamspace/studios/this_studio/messy-mashup/checkpoints/ast_best_phase2.pth',
               map_location=DEVICE)
)
_base.eval()
best_p2_f1 = 0
print("AST loaded from phase 2 checkpoint — skipping retraining")

## 4. CNN Models Training
> **Optimisations:**
> - **`torchaudio.load`** instead of librosa (5–10× faster for short clips)
> - **MelSpectrogram runs on GPU** as a batch transform — removed from `__getitem__` entirely
> - **`time_stretch` removed** (same as AST — random crop replaces it)
> - **AMP/FP16** + **DataParallel** on both T4s
> - **Early stopping** with `patience=5`
> - **Epochs 26 → 15**

### CNN: GPU Mel Transform (shared across all CNN models)
- KEY OPTIMISATION: mel spectrogram is now computed on GPU in the training
- loop over a full batch rather than per-sample on CPU in __getitem__.

In [ ]:
gpu_mel_transform = nn.Sequential(
    T.MelSpectrogram(sample_rate=CNN_SAMPLE_RATE, n_fft=CNN_N_FFT,
                     hop_length=CNN_HOP_LEN, n_mels=CNN_N_MELS),
    T.AmplitudeToDB()
).to(DEVICE)

print("GPU mel transform ready (MelSpectrogram + AmplitudeToDB on CUDA)!")
print(f"  Output shape for 1 sample: "
      f"{gpu_mel_transform(torch.zeros(1, CNN_N_SAMPLES).to(DEVICE)).shape}")

### CNN: Dataset (returns raw audio waveform, NOT spectrogram) ───
- torchaudio.load is ~5-10× faster than librosa for short clips.
- NO mel computation here — done in batch on GPU in the training loop.
- NO time_stretch — random crop offset from different position instead.

In [ ]:
class CNNTrainDataset(Dataset):
    """Returns (raw_waveform [N_SAMPLES], label) — mel computed on GPU."""
    def __init__(self, song_index, noise_files, size=8000):
        self.song_index  = song_index
        self.noise_files = noise_files
        self.size        = size

    def __len__(self): return self.size

    def _fast_load(self, path):
        """torchaudio.load → numpy, resampled to CNN_SAMPLE_RATE, 5s crop."""
        try:
            wav, sr = torchaudio.load(path)                 # shape (C, T)
            if sr != CNN_SAMPLE_RATE:
                wav = torchaudio.functional.resample(wav, sr, CNN_SAMPLE_RATE)
            wav = wav.mean(0).numpy()                        # mono
            if len(wav) < CNN_N_SAMPLES:
                wav = np.pad(wav, (0, CNN_N_SAMPLES - len(wav)))
            # ⚡ random offset crop — augmentation with zero overhead
            start = random.randint(0, max(0, len(wav) - CNN_N_SAMPLES))
            return wav[start : start + CNN_N_SAMPLES].astype(np.float32)
        except Exception:
            return np.zeros(CNN_N_SAMPLES, dtype=np.float32)

    def __getitem__(self, idx):
        genre    = random.choice(GENRES)
        label    = label2id[genre]
        songs    = self.song_index[genre]
        if not songs:
            return torch.zeros(CNN_N_SAMPLES), label
        selected = random.choices(songs, k=4) if len(songs)<4 else random.sample(songs, 4)
        stems    = ['drums.wav', 'bass.wav', 'vocals.wav', 'other.wav']
        mixed    = np.zeros(CNN_N_SAMPLES, dtype=np.float32)

        for i, stem in enumerate(stems):
            y    = self._fast_load(os.path.join(selected[i], stem))
            mixed += y * random.uniform(0.7, 1.3)

        # ESC-50 noise
        if self.noise_files and random.random() < 0.8:
            n    = self._fast_load(random.choice(self.noise_files))
            mixed += n * random.uniform(0.1, 0.4)

        mx = np.max(np.abs(mixed))
        if mx > 0: mixed /= mx
        return torch.tensor(mixed, dtype=torch.float32), label


class CNNValDataset(Dataset):
    """Fixed centre-crop — same val_index as AST."""
    def __init__(self, val_index):
        self.samples = [(label2id[g], sp) for g in GENRES for sp in val_index[g]]

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        label, song_path = self.samples[idx]
        mixed = np.zeros(CNN_N_SAMPLES, dtype=np.float32)
        for stem in ['drums.wav','bass.wav','vocals.wav','other.wav']:
            sp = os.path.join(song_path, stem)
            if not os.path.exists(sp): continue
            try:
                wav, sr = torchaudio.load(sp)
                if sr != CNN_SAMPLE_RATE:
                    wav = torchaudio.functional.resample(wav, sr, CNN_SAMPLE_RATE)
                wav = wav.mean(0).numpy()
                if len(wav) < CNN_N_SAMPLES:
                    wav = np.pad(wav, (0, CNN_N_SAMPLES-len(wav)))
                # centre crop
                start = (len(wav) - CNN_N_SAMPLES) // 2
                mixed += wav[start:start+CNN_N_SAMPLES].astype(np.float32) * 0.25
            except: pass
        mx = np.max(np.abs(mixed))
        if mx > 0: mixed /= mx
        return torch.tensor(mixed, dtype=torch.float32), label

cnn_train_ds = CNNTrainDataset(train_index, noise_files_list, size=8000)
cnn_val_ds   = CNNValDataset(val_index)
cnn_train_loader = DataLoader(cnn_train_ds, batch_size=CNN_BATCH_SIZE, shuffle=True,
                               num_workers=CNN_NUM_WORKERS, pin_memory=True, persistent_workers=True)
cnn_val_loader   = DataLoader(cnn_val_ds,   batch_size=CNN_BATCH_SIZE, shuffle=False,
                               num_workers=CNN_NUM_WORKERS, pin_memory=True)
print(f"CNN Train: {len(cnn_train_ds)} | Val: {len(cnn_val_ds)}")
print(f"Train batches: {len(cnn_train_loader)} | Val batches: {len(cnn_val_loader)}")


#### CNN: Model Architectures

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes=NUM_LABELS):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1,32,3,padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32,64,3,padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(),
            nn.Linear(128,256), nn.ReLU(), nn.Dropout(0.3), nn.Linear(256,num_classes)
        )
    def forward(self, x): return self.classifier(self.features(x))

class AudioResNet(nn.Module):
    def __init__(self, num_classes=NUM_LABELS):
        super().__init__()
        self.resnet = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        orig_w = self.resnet.conv1.weight.clone()
        self.resnet.conv1 = nn.Conv2d(1,64,7,2,3,bias=False)
        with torch.no_grad():
            self.resnet.conv1.weight[:,0] = orig_w.mean(dim=1)
        self.resnet.fc = nn.Linear(self.resnet.fc.in_features, num_classes)
    def forward(self, x): return self.resnet(x)

class AudioEfficientNet(nn.Module):
    def __init__(self, num_classes=NUM_LABELS):
        super().__init__()
        self.model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        orig_w = self.model.features[0][0].weight.clone()
        self.model.features[0][0] = nn.Conv2d(1,32,3,stride=2,padding=1,bias=False)
        with torch.no_grad():
            self.model.features[0][0].weight[:,0] = orig_w.mean(dim=1)
        self.model.classifier[1] = nn.Linear(self.model.classifier[1].in_features, num_classes)
    def forward(self, x): return self.model(x)

for cls in [CustomCNN, AudioResNet, AudioEfficientNet]:
    ps = sum(p.numel() for p in cls().parameters())
    print(f"  {cls.__name__:22s}: {ps:,} params")


### CNN: Unified Training Function (AMP + GPU mel + DataParallel + early stop)

In [ ]:
def train_cnn_model(model, model_name, epochs=CNN_EPOCHS, lr=CNN_LR):
    
    wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name=model_name, reinit=True,
               config=dict(model=model_name, epochs=epochs, lr=lr,
                           batch_size=CNN_BATCH_SIZE, weight_decay=CNN_WEIGHT_DECAY,
                           n_mels=CNN_N_MELS, duration_s=CNN_DURATION,
                           amp=USE_AMP, dataparallel=USE_DDP, patience=CNN_PATIENCE))

    # DataParallel
    if USE_DDP and N_GPUS > 1:
        model = nn.DataParallel(model)
        print(f"  {model_name}: DataParallel across {N_GPUS} GPUs")
    model = model.to(DEVICE)

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=CNN_WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = torch.cuda.amp.GradScaler(enabled=USE_AMP)  # AMP scaler

    os.makedirs('/teamspace/studios/this_studio/messy-mashup/checkpoints/checkpoints', exist_ok=True)
    best_f1, best_epoch, patience_cnt = -1.0, -1, 0

    for epoch in range(epochs):
        # Train
        model.train()
        t_preds, t_labels, t_loss = [], [], 0.0

        for raw, labels in tqdm(cnn_train_loader,
                                desc=f"  [{model_name}] Ep {epoch+1:02d} Train"):
            raw,labels = raw.to(DEVICE), labels.to(DEVICE)
            # Mel on GPU — no Python overhead, batched CUDA kernel
            with torch.no_grad():
                specs = gpu_mel_transform(raw.unsqueeze(1))   # (B,1,128,T)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=USE_AMP):    # FP16 forward
                out  = model(specs)
                loss = criterion(out, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update()
            t_loss += loss.item()
            p = torch.argmax(out, 1)
            t_preds.extend(p.cpu().numpy()); t_labels.extend(labels.cpu().numpy())

        scheduler.step()

        # Validate 
        model.eval()
        v_preds, v_labels, v_loss = [], [], 0.0
        with torch.no_grad():
            for raw, labels in tqdm(cnn_val_loader,
                                    desc=f"  [{model_name}] Ep {epoch+1:02d} Val  "):
                raw, labels = raw.to(DEVICE), labels.to(DEVICE)
                specs = gpu_mel_transform(raw.unsqueeze(1))
                with torch.cuda.amp.autocast(enabled=USE_AMP):
                    out  = model(specs)
                    loss = criterion(out, labels)
                v_loss += loss.item()
                p = torch.argmax(out, 1)
                v_preds.extend(p.cpu().numpy()); v_labels.extend(labels.cpu().numpy())

        # Metrics 
        tf1   = f1_score(t_labels, t_preds, average='macro', zero_division=0)
        vf1   = f1_score(v_labels, v_preds, average='macro', zero_division=0)
        tacc  = accuracy_score(t_labels, t_preds)
        vacc  = accuracy_score(v_labels, v_preds)
        vpc   = f1_score(v_labels, v_preds, average=None, labels=list(range(NUM_LABELS)), zero_division=0)
        tpc   = f1_score(t_labels, t_preds, average=None, labels=list(range(NUM_LABELS)), zero_division=0)

        log = {"epoch": epoch+1,
               "train/loss": t_loss/len(cnn_train_loader), "train/accuracy": tacc, "train/macro_f1": tf1,
               "val/loss":   v_loss/len(cnn_val_loader),   "val/accuracy":   vacc, "val/macro_f1":   vf1,
               "lr": scheduler.get_last_lr()[0]}
        for i, g in enumerate(GENRES):
            log[f"val/f1_{g}"] = float(vpc[i]); log[f"train/f1_{g}"] = float(tpc[i])
        wandb.log(log)

        print(f"  Ep {epoch+1:02d}/{epochs} "
              f"| TL:{t_loss/len(cnn_train_loader):.4f} TF1:{tf1:.4f} "
              f"|| VL:{v_loss/len(cnn_val_loader):.4f} VF1:{vf1:.4f} VAcc:{vacc:.4f}")

        # Checkpoint + early stoppin
        if vf1 > best_f1:
            best_f1, best_epoch, patience_cnt = vf1, epoch, 0
            ckpt = f"/teamspace/studios/this_studio/messy-mashup/checkpoints/checkpoints/{model_name}_best.pth"
            _m   = model.module if hasattr(model, 'module') else model
            torch.save({'epoch': epoch, 'model_state_dict': _m.state_dict(),
                        'val_f1': vf1}, ckpt)
            print(f" Saved {ckpt}  (F1: {best_f1:.4f})")
        else:
            patience_cnt += 1
            if patience_cnt >= CNN_PATIENCE:
                print(f"  Early stopping at epoch {epoch+1} (patience={CNN_PATIENCE})")
                break

    print(f"  Best: epoch {best_epoch+1} | F1: {best_f1:.4f}")
    wandb.run.summary.update({"best_val_f1": best_f1, "best_epoch": best_epoch+1})
    wandb.finish()

    # Return base model (unwrapped)
    return model.module if hasattr(model, 'module') else model

print("train_cnn_model ready (AMP + GPU mel + DataParallel + early stopping)!")


### CNN: CustomCNN — SKIP (checkpoint exists)

In [ ]:
cnn_model1 = CustomCNN()
ckpt1 = torch.load(
    '/teamspace/studios/this_studio/messy-mashup/checkpoints/checkpoints/CustomCNN_best.pth',
    map_location=DEVICE
)
cnn_model1.load_state_dict(ckpt1['model_state_dict'])
cnn_model1.eval()
print(f" CustomCNN loaded from checkpoint (val F1: {ckpt1['val_f1']:.4f})")

### CNN: Train CustomCNN

In [ ]:
cnn_model1 = train_cnn_model(CustomCNN(), "CustomCNN")
cnn_model2 = train_cnn_model(AudioResNet(), "AudioResNet34")
cnn_model3 = train_cnn_model(AudioEfficientNet(), "EfficientNetB0")

## 5. Ensemble Inference & Submission

### Load Best Checkpoints

In [ ]:
_ast_base.load_state_dict(torch.load('/teamspace/studios/this_studio/messy-mashup/checkpoints/ast_best_phase2.pth'))
_ast_base.eval().to(DEVICE)
print(f"AST best Phase 2 loaded | val F1 was: {best_p2_f1:.4f}")

def load_cnn(cls, name):
    m    = cls()
    ckpt = torch.load(f'/teamspace/studios/this_studio/messy-mashup/checkpoints/checkpoints/{name}_best.pth', map_location=DEVICE)
    m.load_state_dict(ckpt['model_state_dict'])
    m.eval().to(DEVICE)
    print(f"{name} loaded | best val F1: {ckpt['val_f1']:.4f}")
    return m

cnn_model1 = load_cnn(CustomCNN,        "CustomCNN")
cnn_model2 = load_cnn(AudioResNet,      "AudioResNet34")
cnn_model3 = load_cnn(AudioEfficientNet,"EfficientNetB0")
print(f"\nEnsemble weights: {ENSEMBLE_WEIGHTS}")


### Individual Val F1 Summary

In [ ]:
val_results = {}

# AST
ap, al = [], []
with torch.no_grad():
    for batch in tqdm(ast_val_loader, desc="AST val"):
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            out = _ast_base(input_values=batch['input_values'].to(DEVICE))
        ap.extend(torch.argmax(out.logits,1).cpu().numpy())
        al.extend(batch['labels'].numpy())
val_results['AST'] = f1_score(al, ap, average='macro', zero_division=0)

# CNNs
for m, name in [(cnn_model1,'CustomCNN'),(cnn_model2,'AudioResNet34'),(cnn_model3,'EfficientNetB0')]:
    cp, cl = [], []
    with torch.no_grad():
        for raw, labels in tqdm(cnn_val_loader, desc=f"{name} val"):
            specs = gpu_mel_transform(raw.unsqueeze(1).to(DEVICE))
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out = m(specs)
            cp.extend(torch.argmax(out,1).cpu().numpy())
            cl.extend(labels.numpy())
    val_results[name] = f1_score(cl, cp, average='macro', zero_division=0)

print("\n" + "="*45)
print("INDIVIDUAL MODEL VAL F1")
print("="*45)
for k, v in val_results.items():
    print(f"  {k:22s}: {v:.4f}  (weight {ENSEMBLE_WEIGHTS.get(k,0):.2f})")


### Inference Helpers

In [ ]:
def ast_predict_proba(audio):
    probs = []
    for _ in range(AST_N_TTA):
        c = ast_normalize(ast_crop_random(audio))
        inp = ast_feature_extractor(c, sampling_rate=AST_SAMPLE_RATE, return_tensors='pt')
        with torch.no_grad():
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                out = _ast_base(input_values=inp['input_values'].to(DEVICE))
            probs.append(torch.softmax(out.logits, 1).cpu().numpy()[0])
    return np.mean(probs, axis=0)

def cnn_predict_proba(audio_path):
    # 3-chunk TTA (start/centre/end) using torchaudio
    try:
        wav, sr = torchaudio.load(audio_path)
        if sr != CNN_SAMPLE_RATE:
            wav = torchaudio.functional.resample(wav, sr, CNN_SAMPLE_RATE)
        wav = wav.mean(0).numpy()
    except:
        wav = np.zeros(CNN_N_SAMPLES)
    L  = len(wav)
    n  = CNN_N_SAMPLES
    if L < n: wav = np.pad(wav,(0,n-L)); L = n
    chunks = [wav[:n], wav[max(0,L//2-n//2):max(0,L//2-n//2)+n], wav[L-n:]]
    chunk_tensors = torch.stack([
        torch.tensor(c[:n] if len(c)>=n else np.pad(c,(0,n-len(c))),
                     dtype=torch.float32) for c in chunks
    ]).unsqueeze(1).to(DEVICE)    # (3,1,N)
    specs = gpu_mel_transform(chunk_tensors.squeeze(1).unsqueeze(1))
    probs = []
    with torch.no_grad():
        with torch.cuda.amp.autocast(enabled=USE_AMP):
            for m in [cnn_model1, cnn_model2, cnn_model3]:
                out = m(specs)
                probs.append(torch.softmax(out,1).cpu().numpy())
    # probs: list of 3 arrays (3,10); average over chunks then return per model
    c1 = np.mean(probs[0], axis=0)
    c2 = np.mean(probs[1], axis=0)
    c3 = np.mean(probs[2], axis=0)
    return c1, c2, c3

print("Inference helpers ready!")


## 6. Generate Submission

In [ ]:
test_df = pd.read_csv(TEST_CSV)
file_map = {}
for f in Path(MASHUPS_PATH).glob("*.wav"):
    nums = ''.join(filter(str.isdigit, f.stem))
    if nums: file_map[int(nums)] = str(f)
print(f"Test samples: {len(test_df)} | Audio files found: {len(file_map)}")

all_ids, all_preds = [], []
for idx in tqdm(range(len(test_df)), desc="Ensemble Inference"):
    row = test_df.iloc[idx]
    try: id_int = int(row['id'])
    except: id_int = int(''.join(filter(str.isdigit, str(row['id']))))
    audio_path = file_map.get(id_int, os.path.join(MASHUPS_PATH, str(row['id'])))

    try:
        ast_audio = ast_load_audio(audio_path)
        p_ast           = ast_predict_proba(ast_audio)
        p_c1, p_c2, p_c3 = cnn_predict_proba(audio_path)
        ep = (ENSEMBLE_WEIGHTS['AST']            * p_ast +
              ENSEMBLE_WEIGHTS['CustomCNN']      * p_c1  +
              ENSEMBLE_WEIGHTS['AudioResNet34']  * p_c2  +
              ENSEMBLE_WEIGHTS['EfficientNetB0'] * p_c3)
        pred = id2label[int(np.argmax(ep))]
    except Exception as e:
        pred = 'rock'

    all_ids.append(row['id']); all_preds.append(pred)

submission = pd.DataFrame({'id': all_ids, 'genre': all_preds})
submission.to_csv('/teamspace/studios/this_studio/messy-mashup/checkpoints/submission_ensemble.csv', index=False)
print(f"\nSubmission saved!  Shape: {submission.shape}")
print(submission['genre'].value_counts())


## Confusion Matrix on Val Set

In [ ]:
val_true, val_ens_preds = [], []
for genre in GENRES:
    for sp in tqdm(val_index[genre], desc=f"Val {genre}"):
        try:
            # AST
            ast_audio = np.zeros(AST_MAX_LENGTH, dtype=np.float32)
            for stem in REQUIRED_STEMS:
                stp = os.path.join(sp, stem)
                if os.path.exists(stp):
                    a = ast_load_audio(stp)
                    a = ast_crop_center(a)
                    ast_audio += a
            ast_audio = ast_normalize(ast_audio)
            inp = ast_feature_extractor(ast_audio, sampling_rate=AST_SAMPLE_RATE, return_tensors='pt')
            with torch.no_grad():
                with torch.cuda.amp.autocast(enabled=USE_AMP):
                    out_ast = torch.softmax(_ast_base(input_values=inp['input_values'].to(DEVICE)).logits,1).cpu().numpy()[0]

            # CNNs — build mixed waveform
            mixed = np.zeros(CNN_N_SAMPLES, dtype=np.float32)
            for stem in ['drums.wav','bass.wav','vocals.wav','other.wav']:
                stp = os.path.join(sp, stem)
                if os.path.exists(stp):
                    try:
                        wav, sr = torchaudio.load(stp)
                        if sr!=CNN_SAMPLE_RATE: wav=torchaudio.functional.resample(wav,sr,CNN_SAMPLE_RATE)
                        wav=wav.mean(0).numpy()
                        if len(wav)<CNN_N_SAMPLES: wav=np.pad(wav,(0,CNN_N_SAMPLES-len(wav)))
                        mixed+=wav[:CNN_N_SAMPLES]*0.25
                    except: pass
            mx=np.max(np.abs(mixed)); mixed=mixed/mx if mx>0 else mixed
            spec=gpu_mel_transform(torch.tensor(mixed).unsqueeze(0).unsqueeze(0).to(DEVICE))
            with torch.no_grad():
                with torch.cuda.amp.autocast(enabled=USE_AMP):
                    oc1=torch.softmax(cnn_model1(spec),1).cpu().numpy()[0]
                    oc2=torch.softmax(cnn_model2(spec),1).cpu().numpy()[0]
                    oc3=torch.softmax(cnn_model3(spec),1).cpu().numpy()[0]

            ep=(ENSEMBLE_WEIGHTS['AST']*out_ast + ENSEMBLE_WEIGHTS['CustomCNN']*oc1 +
                ENSEMBLE_WEIGHTS['AudioResNet34']*oc2 + ENSEMBLE_WEIGHTS['EfficientNetB0']*oc3)
            val_true.append(label2id[genre])
            val_ens_preds.append(int(np.argmax(ep)))
        except: pass

ens_f1 = f1_score(val_true, val_ens_preds, average='macro', zero_division=0)
print(f"\nEnsemble Val Macro F1: {ens_f1:.4f}")
print(classification_report(val_true, val_ens_preds, target_names=GENRES, zero_division=0))

cm  = confusion_matrix(val_true, val_ens_preds)
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=GENRES, yticklabels=GENRES, ax=ax)
ax.set_title(f"Ensemble Confusion Matrix  (Val Macro F1: {ens_f1:.4f})", fontweight='bold')
ax.set_xlabel("Predicted"); ax.set_ylabel("True")
plt.tight_layout()
plt.savefig('/teamspace/studios/this_studio/messy-mashup/checkpoints/ensemble_cm.png', dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# WandB Ensemble Summary Run
wandb.init(project=WANDB_PROJECT, entity=WANDB_ENTITY, name="Ensemble_Summary",
           config=dict(ensemble_weights=ENSEMBLE_WEIGHTS, n_tta=AST_N_TTA))
for k, v in val_results.items():
    wandb.log({f"individual_val_f1/{k}": v})
wandb.log({"ensemble/val_macro_f1": ens_f1,
           "ensemble/improvement": ens_f1 - max(val_results.values())})
pc = f1_score(val_true, val_ens_preds, average=None, labels=list(range(NUM_LABELS)), zero_division=0)
for i, g in enumerate(GENRES):
    wandb.log({f"ensemble/val_f1_{g}": float(pc[i])})

fig, ax = plt.subplots(figsize=(9, 4))
names  = list(val_results.keys()) + ["Ensemble"]
scores = list(val_results.values()) + [ens_f1]
bars   = ax.bar(names, scores, color=['#3498db','#e74c3c','#2ecc71','#f39c12','#9b59b6'], alpha=0.85)
ax.set_ylim(0, 1.05)
for b, s in zip(bars, scores):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f"{s:.4f}",
            ha='center', fontsize=10, fontweight='bold')
ax.set_title("Val Macro F1: All Models vs Ensemble", fontweight='bold')
ax.set_ylabel("Macro F1")
plt.tight_layout()
plt.savefig('/teamspace/studios/this_studio/messy-mashup/checkpoints/model_comparison.png', dpi=100, bbox_inches='tight')
wandb.log({"model_comparison": wandb.Image('/teamspace/studios/this_studio/messy-mashup/checkpoints/model_comparison.png')})
plt.show()
wandb.run.summary["ensemble_val_f1"] = ens_f1
wandb.finish()


## 7. Final Summary

In [ ]:
print("  DONE!")
print()
print("Individual Val Macro F1:")
for k, v in val_results.items():
    print(f"  {k:22s}: {v:.4f}  (weight {ENSEMBLE_WEIGHTS.get(k,0):.2f})")
print(f"  {'Ensemble':22s}: {ens_f1:.4f}")
print()
print("Submission: /teamspace/studios/this_studio/messy-mashup/checkpoints/submission_ensemble.csv")
print("=" * 60)
